# Station 3: Signal — 15 points

Practice Morse timing, waveform inspection, and simple spectral reasoning. The supplied WAV is generated locally and contains a real audible practice message.

**Scope:** These are practice scenarios inferred from `urtuu.md`, not official competition tasks.

**Offline setup:** Python 3 kernel; all exercise code uses the standard library. Run cells from top to bottom. No downloads or internet are required.

**How to study:** Read the worked example, then attempt the three exercises before opening the solution section. Use a 30–40 minute timer. Record any recovered fragment exactly.

## Learn it visually



**How to read this:** The horizontal axis represents time. Longer green bars last three times as long as short bars. The larger empty spaces separate letters. This is a timing diagram, not the rapidly oscillating waveform of the tone itself.

**Try before continuing:** If one dot lasts 0.1 seconds, how long are a dash and a letter gap?

<details><summary>Show the answer</summary>

Both last 0.3 seconds. A word gap lasts 0.7 seconds.

</details>


## Mini-course: hearing and seeing hidden information

**What the guideline confirms:** A message is hidden in sound; listening, viewing, and analysis are involved. Morse and spectrogram puzzles are possibilities, not confirmed formats.

### Lesson 1 — Sound as sampled numbers
A digital recording stores amplitude measurements at regular intervals. Amplitude describes signal strength; frequency describes how quickly a sound oscillates and relates to pitch. A sample rate of 8,000 Hz means 8,000 measurements each second. A waveform plots amplitude against time. It is useful for spotting pulses, pauses, and repeated patterns.

### Lesson 2 — Morse uses duration
A dot is one time unit, a dash three. Gaps distinguish marks, letters, and words. Three short marks are S; three long marks are O. Therefore `... --- ...` spells SOS. First estimate the short duration; then compare other sounds and gaps to it. Do not decode every pause as a new letter.

### Lesson 3 — A spectrogram shows another view
A spectrogram usually shows time horizontally, frequency vertically, and intensity with brightness or color. A sustained tone becomes a horizontal line. Changing tones can form patterns or even letters. Stereo audio has separate left and right channels: inspecting each independently can reveal something obscured in the mix.

### Lesson 4 — Sampling limits
To distinguish a tone reliably, the sampling rate must exceed twice its frequency. Half the sampling rate is the Nyquist frequency. Higher tones can masquerade as lower tones, called aliasing. For a transform of N samples at sample rate Fs, frequency bin k corresponds to k × Fs / N. You only need this formula for the frequency exercise below.

| Possible task (prediction) | First action |
|---|---|
| Short/long beeps | Mark durations and gaps; try Morse |
| Reversed or slowed speech | Try reverse playback or a speed adjustment |
| Strange sustained tones | Inspect a spectrogram |
| Different sounds in each ear | Inspect left and right channels separately |

**Learn first:** waveform versus spectrogram, Morse timing, and listening systematically. **Extra practice:** frequency bins and aliasing. The notebook includes audio and a waveform exercise; spectrogram and reverse-playback tasks are suggested extensions.


## Worked example: timing
One timing unit is a dot. A dash lasts three units. Silence is usually one unit between marks, three between letters, and seven between words. A threshold converts noisy measurements into mark/gap categories. Always estimate the unit from the data.

The inline player below is embedded in the notebook. The same audio is also supplied as `signal_practice.wav`.

<audio controls src="..."></audio>

If your notebook frontend blocks inline audio, open the neighboring WAV file with a local player.

In [ ]:
MORSE = {"...":"S", "---":"O", "..":"I", "--.":"G", "-.":"N", ".-":"A", ".-..":"L", ".":"E", "-":"T"}
print(" ".join(MORSE[x] for x in "... --- ...".split()))

## Your exercises

1. **Listen and decode (8 minutes):** Decode the supplied audio, first by ear or by marking short/long pulses. The recovered word is a fragment.
2. **Measured durations (10 minutes):** Decode the `events` sequence below. Durations are milliseconds; `on` means sound. Estimate the dot unit, choose a threshold, and handle letter boundaries.
3. **Frequency puzzle (8 minutes):** At a sample rate of 8,000 Hz with an 800-sample DFT, strong peaks appear at bins 60 and 120. What frequencies are they? What is the Nyquist frequency? Can a 5,000 Hz tone be represented unambiguously at this rate?

Optional: explain how a spectrogram differs from a waveform.

In [ ]:
events = [("on",100),("off",100),("on",100),("off",100),("on",100),("off",300),
          ("on",300),("off",100),("on",300),("off",100),("on",300),("off",300),
          ("on",100),("off",100),("on",100),("off",100),("on",100)]
# Your work here

---
# Solutions — stop here until you have attempted the exercises
Compare both your answer and your reasoning. Rerun the practice with different inputs after studying the solutions.

In [ ]:
print("Audio fragment:", "".join(MORSE[x] for x in "... .. --. -. .- .-..".split()))
letters, current = [], ""
for state, duration in events:
    if state == "on":
        current += "." if duration < 200 else "-"
    elif duration >= 250:
        letters.append(current); current = ""
if current: letters.append(current)
print("Measured message:", "".join(MORSE[x] for x in letters))
print("Frequencies:", [k*8000/800 for k in (60,120)])
print("Nyquist:", 8000/2, "Hz")

**Answers:** The audio says SIGNAL; the durations say SOS. The peaks are 600 Hz and 1,200 Hz. Nyquist is 4,000 Hz. A sampled 5,000 Hz tone aliases to 3,000 Hz at this sample rate, so it is ambiguous. A waveform shows amplitude over time; a spectrogram shows frequency content over time.

## Visualize the actual audio without plotting packages
The next cell produces an inline SVG amplitude envelope using the standard library and the notebook display protocol.

In [ ]:
import wave, struct
from pathlib import Path
class SVG:
    def __init__(self, text): self.text=text
    def _repr_svg_(self): return self.text
audio_path = Path("signal_practice.wav")
if not audio_path.exists(): audio_path = Path("preparation/signal_practice.wav")
with wave.open(str(audio_path), "rb") as f:
    data = struct.unpack("<"+"h"*f.getnframes(), f.readframes(f.getnframes()))
step=160
peaks=[max(abs(x) for x in data[i:i+step])/32768 for i in range(0,len(data),step)]
points=" ".join(f"{20+i*700/max(1,len(peaks)-1):.1f},{145-p*200:.1f}" for i,p in enumerate(peaks))
SVG(f'<svg xmlns="http://www.w3.org/2000/svg" width="750" height="190"><rect width="750" height="190" fill="#101d32"/><text x="20" y="25" fill="white">Audio amplitude envelope — time →</text><polyline points="{points}" fill="none" stroke="#5eead4" stroke-width="2"/></svg>')